In [1]:

# %%
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Locate project root
project_root = Path.cwd().parents[2]

# Add project root to Python's module search path
sys.path.insert(0, str(project_root))

# Import reusable modules
from src.data_loader import load_cv_data
import src.feature_engineering as fe

# Load the latest feature-engineering implementation
importlib.reload(fe)

print("Project root:", project_root)
print("Setup completed.")


Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Setup completed.


In [2]:

# %%
fold_datasets = load_cv_data(project_root)

# Convert skill IDs into sets
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


# Convert skill IDs in every fold
for train_df, val_df in fold_datasets:

    train_df["skill_set"] = train_df["skill_id"].apply(
        convert_skill_id
    )

    val_df["skill_set"] = val_df["skill_id"].apply(
        convert_skill_id
    )


# Display fold sizes
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )


Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [3]:

# %%
fold_train_df, fold_val_df = fold_datasets[0]

print("Original skill IDs:")
print(fold_train_df["skill_id"].head(10).tolist())

print("\nConverted skill sets:")
print(fold_train_df["skill_set"].head(10).tolist())

print("\nData type:")
print(type(fold_train_df["skill_set"].iloc[0]))

print("\nMissing skill IDs:")
print(fold_train_df["skill_id"].isna().sum())

print("\nSkill-set size distribution:")
print(
    fold_train_df["skill_set"]
    .apply(len)
    .value_counts()
    .sort_index()
)


Original skill IDs:
['2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_48_77', '2_48_79', '2_37_48', '2_48_79']

Converted skill sets:
[{'2', '37', '70'}, {'2', '37', '70'}, {'2', '37', '70'}, {'2', '37', '70'}, {'2', '37', '70'}, {'2', '37', '70'}, {'2', '37', '48', '77'}, {'2', '79', '48'}, {'2', '37', '48'}, {'2', '79', '48'}]

Data type:
<class 'set'>

Missing skill IDs:
11163

Skill-set size distribution:
skill_set
0     11163
1    136536
2     25677
3      2885
4      1063
Name: count, dtype: int64


In [4]:

# %%
# Target question
target_skills = {"A", "B", "C"}

# Previous questions
previous_skill_sets = [
    {"A", "B", "C"},
    {"A", "B"},
    {"A", "D"},
    {"D", "E"},
    {"A"},
    set()
]

# Calculate fractional overlap weights
for previous_skills in previous_skill_sets:

    if len(previous_skills) == 0:
        weight = 0.0
    else:
        shared_skills = previous_skills.intersection(
            target_skills
        )

        weight = len(shared_skills) / len(previous_skills)

    print(
        f"Previous skills: {previous_skills}, "
        f"Weight: {weight:.2f}"
    )


Previous skills: {'B', 'A', 'C'}, Weight: 1.00
Previous skills: {'A', 'B'}, Weight: 1.00
Previous skills: {'A', 'D'}, Weight: 0.50
Previous skills: {'E', 'D'}, Weight: 0.00
Previous skills: {'A'}, Weight: 1.00
Previous skills: set(), Weight: 0.00


In [5]:
# %%
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

print(
    "Fractional function available:",
    hasattr(
        fe,
        "add_fractional_same_skill_history_features"
    )
)

import inspect

print(
    "Function signature:",
    inspect.signature(
        fe.add_fractional_same_skill_history_features
    )
)

Fractional function available: True
Function signature: (data, alpha=10, prior_data=None)


In [6]:

# %%
test_data = pd.DataFrame({
    "user_id": [1, 1, 1, 1],
    "order_id": [1, 2, 3, 4],
    "skill_id": ["A_B", "B_C_D", "C_D", "A_C"],
    "correct": [1, 0, 1, 0]
})

test_data["skill_set"] = test_data["skill_id"].apply(
    convert_skill_id
)

test_features = fe.add_fractional_same_skill_history_features(
    test_data,
    alpha=10,
    prior_data=test_data
)

display(
    test_features[
        [
            "order_id",
            "skill_id",
            "correct",
            "weighted_same_skill_previous_interactions",
            "weighted_same_skill_previous_correct",
            "weighted_same_skill_global_prior",
            "weighted_same_skill_smoothed_accuracy"
        ]
    ]
)


,order_id,skill_id,correct,weighted_same_skill_previous_interactions,weighted_same_skill_previous_correct,weighted_same_skill_global_prior,weighted_same_skill_smoothed_accuracy
0,1,A_B,1,0.000000,0.0,0.50,0.50
1,2,B_C_D,0,0.500000,0.5,1.00,1.00
2,3,C_D,1,0.666667,0.0,0.00,0.00
3,4,A_C,0,1.333333,1.0,0.75,0.75


In [7]:

# %%
# Change the outcome of the final question
modified_test_data = test_data.copy()

modified_test_data.loc[
    modified_test_data["order_id"] == 4,
    "correct"
] = 1

modified_features = fe.add_fractional_same_skill_history_features(
    modified_test_data,
    alpha=10,
    prior_data=modified_test_data
)

feature_columns = [
    "weighted_same_skill_previous_interactions",
    "weighted_same_skill_previous_correct",
    "weighted_same_skill_global_prior",
    "weighted_same_skill_smoothed_accuracy"
]

# Compare original and modified features for Q4
original_q4 = test_features.loc[
    test_features["order_id"] == 4,
    feature_columns
].reset_index(drop=True)

modified_q4 = modified_features.loc[
    modified_features["order_id"] == 4,
    feature_columns
].reset_index(drop=True)

print("Original Q4 features:")
display(original_q4)

print("Modified Q4 features:")
display(modified_q4)

print(
    "Features unchanged:",
    np.allclose(
        original_q4.to_numpy(),
        modified_q4.to_numpy()
    )
)


Original Q4 features:


,weighted_same_skill_previous_interactions,weighted_same_skill_previous_correct,weighted_same_skill_global_prior,weighted_same_skill_smoothed_accuracy
0,1.333333,1.0,0.75,0.75


Modified Q4 features:


,weighted_same_skill_previous_interactions,weighted_same_skill_previous_correct,weighted_same_skill_global_prior,weighted_same_skill_smoothed_accuracy
0,1.333333,1.0,0.75,0.75


Features unchanged: True


In [8]:

# %%
v22_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    print(f"\nProcessing Fold {fold_number}...")

    # Generate training features
    v22_train_df = fe.add_fractional_same_skill_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features
    v22_val_df = fe.add_fractional_same_skill_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v22_fold_datasets.append(
        (v22_train_df, v22_val_df)
    )

    print("Training features:", v22_train_df.shape)
    print("Validation features:", v22_val_df.shape)

print("\nV2.2 feature generation completed.")



Processing Fold 1...
Training features: (177324, 11)
Validation features: (42659, 11)

Processing Fold 2...
Training features: (176273, 11)
Validation features: (43710, 11)

Processing Fold 3...
Training features: (170186, 11)
Validation features: (49797, 11)

Processing Fold 4...
Training features: (178311, 11)
Validation features: (41672, 11)

Processing Fold 5...
Training features: (177838, 11)
Validation features: (42145, 11)

V2.2 feature generation completed.


In [9]:

# %%
v22_feature_columns = [
    "weighted_same_skill_previous_interactions",
    "weighted_same_skill_previous_correct",
    "weighted_same_skill_global_prior",
    "weighted_same_skill_smoothed_accuracy"
]

for fold_number, (train_df, val_df) in enumerate(
    v22_fold_datasets, start=1
):

    print(f"\n========== FOLD {fold_number} ==========")

    for name, df in [
        ("Training", train_df),
        ("Validation", val_df)
    ]:

        print(f"\n{name}:")

        print(
            "Missing values:",
            df[v22_feature_columns].isna().sum().sum()
        )

        print(
            "Invalid historical counts:",
            (
                df["weighted_same_skill_previous_correct"]
                >
                df["weighted_same_skill_previous_interactions"]
            ).sum()
        )

        print(
            "Global prior range:",
            df["weighted_same_skill_global_prior"].min(),
            df["weighted_same_skill_global_prior"].max()
        )

        print(
            "Smoothed accuracy range:",
            df["weighted_same_skill_smoothed_accuracy"].min(),
            df["weighted_same_skill_smoothed_accuracy"].max()
        )

        print(
            "Negative fractional counts:",
            (
                df["weighted_same_skill_previous_interactions"] < 0
            ).sum()
        )



========== FOLD 1 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

========== FOLD 2 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

========== FOLD 3 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0
Negative fractional counts: 0

=========

In [10]:

# %%
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    f1_score
)

feature_columns = [
    "weighted_same_skill_smoothed_accuracy"
]

v22_models = []
v22_results = []

for fold_number, (train_data, val_data) in enumerate(
    v22_fold_datasets, start=1
):

    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    # Same preprocessing and model as previous versions
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    # Train using training data only
    model.fit(X_train, y_train)

    # Predict probabilities
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Convert probabilities into binary predictions
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Calculate validation metrics
    ll = log_loss(y_val, y_pred_proba, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred_proba)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    v22_models.append(model)

    v22_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs,
        "accuracy": acc,
        "f1_score": f1
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}, "
        f"Accuracy = {acc:.4f}, "
        f"F1 Score = {f1:.4f}"
    )


Fold 1: Log Loss = 0.5917, Brier Score = 0.2025, Accuracy = 0.6966, F1 Score = 0.7991
Fold 2: Log Loss = 0.5853, Brier Score = 0.1998, Accuracy = 0.7045, F1 Score = 0.8084
Fold 3: Log Loss = 0.5984, Brier Score = 0.2057, Accuracy = 0.6841, F1 Score = 0.7890
Fold 4: Log Loss = 0.5907, Brier Score = 0.2020, Accuracy = 0.6972, F1 Score = 0.8031
Fold 5: Log Loss = 0.5859, Brier Score = 0.2001, Accuracy = 0.7000, F1 Score = 0.8036


In [11]:

# %%
v22_results_df = pd.DataFrame(v22_results)

print("\nV2.2 Cross-Validation Results")
display(v22_results_df)

print("\nAverage Performance:")

for metric in [
    "log_loss",
    "brier_score",
    "accuracy",
    "f1_score"
]:
    print(
        f"Average {metric}: "
        f"{v22_results_df[metric].mean():.4f}"
    )



V2.2 Cross-Validation Results


,fold,log_loss,brier_score,accuracy,f1_score
0,1,0.591666,0.202490,0.696570,0.799125
1,2,0.585271,0.199755,0.704461,0.808361
2,3,0.598384,0.205657,0.684138,0.789017
3,4,0.590712,0.202022,0.697207,0.803102
4,5,0.585879,0.200089,0.699988,0.803573



Average Performance:
Average log_loss: 0.5904
Average brier_score: 0.2020
Average accuracy: 0.6965
Average f1_score: 0.8006


In [12]:
# Collect out-of-fold predictions for V2.2

v22_predictions = []

for fold_number, (train_data, val_data) in enumerate(
    v22_fold_datasets, start=1
):

    # Retrieve the already-trained model for this fold
    model = v22_models[fold_number - 1]

    # Generate validation probabilities
    X_val = val_data[feature_columns]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual outcomes and predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_data["user_id"].values,
        "order_id": val_data["order_id"].values,
        "fold": fold_number,
        "y_true": val_data["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V2.2"
    })

    v22_predictions.append(fold_predictions)

# Combine predictions from all five folds
v22_predictions_df = pd.concat(
    v22_predictions,
    ignore_index=True
)

print("Prediction collection completed.")
print("Total predictions:", len(v22_predictions_df))
display(v22_predictions_df.head())

Prediction collection completed.
Total predictions: 219983


,user_id,order_id,fold,y_true,y_pred_proba,model
0,21825,21616385,1,0,0.452860,V2.2
1,21825,21616478,1,1,0.407203,V2.2
2,21825,21616677,1,1,0.465591,V2.2
3,21825,21616919,1,1,0.515810,V2.2
4,21825,21671620,1,0,0.529583,V2.2


In [13]:
print("Total rows:", len(v22_predictions_df))

print("\nMissing values:")
print(v22_predictions_df.isna().sum())

print("\nFold distribution:")
print(v22_predictions_df["fold"].value_counts().sort_index())

print("\nProbability range:")
print(
    v22_predictions_df["y_pred_proba"].min(),
    v22_predictions_df["y_pred_proba"].max()
)

print("\nActual outcome distribution:")
print(v22_predictions_df["y_true"].value_counts())

Total rows: 219983

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Probability range:
0.0915859435892348 0.9236114224985209

Actual outcome distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64


In [14]:
prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

v22_prediction_path = prediction_dir / "v2_2_predictions.csv"

v22_predictions_df.to_csv(
    v22_prediction_path,
    index=False
)

print("V2.2 predictions saved successfully.")
print("File:", v22_prediction_path)

V2.2 predictions saved successfully.
File: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v2_2_predictions.csv


In [15]:
feature_dir = (
    project_root / "data" / "processed" / "engineered_features"
    / "group_2_same_skill_history" / "v2.2"
)
feature_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "weighted_same_skill_smoothed_accuracy"
]

for fold_number, (train_df, val_df) in enumerate(v22_fold_datasets, start=1):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv", index=False
    )
    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv", index=False
    )
    print(f"Fold {fold_number} saved.")

print("V2.2 features saved successfully!")

Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
V2.2 features saved successfully!
